# BraTS 2021 — Colab download + unzip-to-Drive (streaming, no home upload)

Downloads the dataset **on Colab** (fast), saves the compressed archive to Drive, then
**unzips it in batches of 100 cases** — moving each batch to Drive and deleting the local copy —
so Colab's ~100 GB disk never fills while the full ~114 GB unzipped set is built in Drive.

```
Kaggle -> Colab /content -> [ batch of 100:  unzip -> move to Drive -> delete local ] -> repeat
```

Ends up in `Drive/MyDrive/capstone/`:
```
  brats-2021-task1.zip            compressed source  (~13 GB)
  unzipped/BraTS2021_XXXXX/*.nii  uncompressed data  (~114 GB)
```

**Every stage skips work already done** — safe to re-run after a disconnect:
- download: reused from `/content` or Drive; only fetched from Kaggle if neither has it
- extract: skipped if the cases are already extracted locally
- unzip: cases already in Drive are skipped (resumable)

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
CAPSTONE = '/content/drive/MyDrive/capstone'
UNZIP_DRIVE = os.path.join(CAPSTONE, 'unzipped')
os.makedirs(UNZIP_DRIVE, exist_ok=True)
print('Drive mounted.')
print('  capstone     ->', CAPSTONE)
print('  unzipped dir ->', UNZIP_DRIVE)

## 2. Add your Kaggle API token

Only needed the first time (when the archive isn't in Drive yet). Get it once:
**kaggle.com → avatar → Settings → API → ‘Create New Token’** → downloads `kaggle.json`.
Run the cell, then pick that `kaggle.json` in the dialog. (You can skip this cell if the archive
is already in your Drive.)

In [ ]:
from google.colab import files
print('Upload your kaggle.json ...')
files.upload()  # choose kaggle.json

!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip -q install kaggle
print('Kaggle API ready')

## 3. Get the compressed archive (skips download if it already exists)

Order of preference: already on Colab → copy from Drive → (last resort) download from Kaggle.
So once the `.zip` is in your Drive, this never hits Kaggle again.

In [ ]:
import glob, os, shutil

%cd /content
DRIVE_ZIP = os.path.join(CAPSTONE, 'brats-2021-task1.zip')

local = glob.glob('/content/*.zip')
if local:
    print('archive already on Colab:', local[0], '-> skipping download')
elif os.path.exists(DRIVE_ZIP):
    print('archive found in Drive -> copying to Colab (no Kaggle download) ...')
    shutil.copy(DRIVE_ZIP, '/content/brats-2021-task1.zip')
else:
    print('downloading from Kaggle ...')
    !kaggle datasets download -d dschettler8845/brats-2021-task1

zpath = glob.glob('/content/*.zip')[0]
print('archive:', zpath, round(os.path.getsize(zpath) / 1024**3, 1), 'GB')

# save compressed archive to Drive if it's not already there (idempotent)
if os.path.exists(DRIVE_ZIP) and os.path.getsize(DRIVE_ZIP) == os.path.getsize(zpath):
    print('compressed archive already in Drive.')
else:
    print('saving compressed archive to Drive ...')
    shutil.copy(zpath, DRIVE_ZIP)
    print('saved ->', DRIVE_ZIP)

## 4. Extract the archive on Colab (skips if already extracted)

Gives the per-case `.nii.gz` folders locally (~13 GB, fits). They stay on Colab as the *source*
for unzipping; only the unzipped output is streamed to Drive.

In [ ]:
import zipfile, glob, os, tarfile

RAW = '/content/brats_raw'
os.makedirs(RAW, exist_ok=True)

existing = [d for d in glob.glob(os.path.join(RAW, '**', 'BraTS2021_*'), recursive=True) if os.path.isdir(d)]
if existing:
    case_parent = os.path.dirname(sorted(existing)[0])
    print('already extracted -> skipping extraction.')
else:
    zpath = glob.glob('/content/*.zip')[0]
    print('extracting', zpath, '...')
    with zipfile.ZipFile(zpath) as z:
        z.extractall(RAW)
    for t in glob.glob(os.path.join(RAW, '**', '*.tar'), recursive=True):
        print('untar', t)
        with tarfile.open(t) as tf:
            tf.extractall(RAW)
    existing = [d for d in glob.glob(os.path.join(RAW, '**', 'BraTS2021_*'), recursive=True) if os.path.isdir(d)]
    case_parent = os.path.dirname(sorted(existing)[0])

n = len(glob.glob(os.path.join(case_parent, 'BraTS2021_*')))
print('compressed cases at:', case_parent, '| count:', n)

## 5. Unzip in batches of 100 → move each batch to Drive → delete local → repeat

Peak Colab disk use stays low (source ~13 GB + one batch ~9 GB). Runs ~1–2 hrs total (Drive-write
bound). **If it disconnects, just re-run this cell** — finished cases are skipped.

To do only a subset, set e.g. `cases = cases[:600]` below.

In [ ]:
import glob, gzip, os, shutil, time

SRC = case_parent
STAGE = '/content/unzip_stage'
BATCH = 100

def done_in_drive(name):
    d = os.path.join(UNZIP_DRIVE, name)
    return os.path.isdir(d) and len(glob.glob(os.path.join(d, '*.nii'))) >= 5

def unzip_case(case_dir, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    for gz in sorted(glob.glob(os.path.join(case_dir, '*.nii.gz'))):
        out = os.path.join(out_dir, os.path.basename(gz)[:-3])   # strip .gz
        with gzip.open(gz, 'rb') as fi, open(out, 'wb') as fo:
            shutil.copyfileobj(fi, fo, length=8 * 1024 * 1024)

cases = sorted(d for d in glob.glob(os.path.join(SRC, 'BraTS2021_*')) if os.path.isdir(d))
# cases = cases[:600]   # <- uncomment to unzip only a subset
print(f'{len(cases)} cases total; batch size {BATCH}')
t0 = time.time(); new = 0

for i in range(0, len(cases), BATCH):
    batch = cases[i:i + BATCH]
    if os.path.isdir(STAGE): shutil.rmtree(STAGE)
    os.makedirs(STAGE, exist_ok=True)

    # 1) unzip this batch into local staging (skip cases already in Drive)
    staged = []
    for c in batch:
        name = os.path.basename(c)
        if done_in_drive(name):
            continue
        out_dir = os.path.join(STAGE, name)
        unzip_case(c, out_dir)
        staged.append((name, out_dir))

    # 2) move the unzipped batch to Drive (copy + remove local)
    for name, out_dir in staged:
        dst = os.path.join(UNZIP_DRIVE, name)
        if os.path.isdir(dst): shutil.rmtree(dst)   # clear any partial
        shutil.move(out_dir, dst)
        new += 1

    # 3) delete local staging for this batch
    if os.path.isdir(STAGE): shutil.rmtree(STAGE)
    print(f'  {min(i + BATCH, len(cases))}/{len(cases)} cases  (+{len(staged)} this batch, {new} new total, {time.time() - t0:.0f}s)', flush=True)

print(f'DONE: {new} cases unzipped to Drive in {time.time() - t0:.0f}s')

## 6. Verify the unzipped data in Drive

In [ ]:
import glob, os

cases = sorted(glob.glob(os.path.join(UNZIP_DRIVE, 'BraTS2021_*')))
print('unzipped case folders in Drive:', len(cases))
if cases:
    sample = cases[0]
    fs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(sample, '*.nii')))
    print('sample', os.path.basename(sample), '->', fs)